In [8]:
from pathlib import Path
import xarray as xr
import sys

sys.path.append(str(Path("../src").resolve()))
from radar_lwc.retrieval.gas_attenuation import specific_gas_attenuation, alpha_field

In [9]:
Processed_Dataset = "20261002_pallas_interpolation_CLARA"

In [10]:
in_path = Path(f"../data/processed/{Processed_Dataset}/aligned_LWP2.nc")
ds = xr.open_dataset(in_path)

In [11]:
alpha_W = alpha_field(
    P_field=ds["P_cloud"],
    T_field=ds["T_cloud"],
    AH_field=ds["AH_cloud"],
    freq_ghz=94.0,
)

alpha_Ka = alpha_field(
    P_field=ds["P_cloud"],
    T_field=ds["T_cloud"],
    AH_field=ds["AH_cloud"],
    freq_ghz=35.0,
)

In [12]:
print(alpha_W)
print(alpha_Ka)

print(f"W-band:  {float(alpha_W.min()):.3f} - {float(alpha_W.max()):.3f} dB/km")
print(f"Ka-band: {float(alpha_Ka.min()):.3f} - {float(alpha_Ka.max()):.3f} dB/km")

<xarray.DataArray 'alpha_94GHz' (Time: 35268, range: 442)> Size: 125MB
array([[0.42590756, 0.42373918, 0.42155323, ...,        nan,        nan,
               nan],
       [0.425927  , 0.4237584 , 0.42157224, ...,        nan,        nan,
               nan],
       [0.42594634, 0.42377754, 0.42159116, ...,        nan,        nan,
               nan],
       ...,
       [0.52571775, 0.52389987, 0.52207209, ...,        nan,        nan,
               nan],
       [0.52571775, 0.52389987, 0.52207209, ...,        nan,        nan,
               nan],
       [0.52571775, 0.52389987, 0.52207209, ...,        nan,        nan,
               nan]], shape=(35268, 442))
Coordinates:
  * Time     (Time) datetime64[ns] 282kB 2026-10-02T00:46:20.572915077 ... 20...
  * range    (range) float64 4kB 104.3 119.3 134.2 ... 1.094e+04 1.097e+04
Attributes:
    units:      dB km-1
    long_name:  Gaseous attenuation at 94.0 GHz
    model:      ITU-R P.676
<xarray.DataArray 'alpha_35GHz' (Time: 35268, range

In [13]:
ds_alpha = xr.Dataset(
    {
        "alpha_W": alpha_W,
        "alpha_Ka": alpha_Ka,
    },
    attrs={
        "description": "Gas attenuation computed with ITU-R P.676-13",
        "created_by": "02_05_gas_attenuation.ipynb",
    },
)

ds_alpha.to_netcdf(f"../data/processed/{Processed_Dataset}/gas_attenuation.nc")